<a href="https://colab.research.google.com/github/SoupBoi1/TLS-Fingerprinting-Malicious-Detection/blob/main/TLSMaliciousDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#TLSFingerprintingMaliciousDetectionNotebook1

#author Matt Lutjen, Brady Bangasser, Sudipta Halder,

#Importing data and libs

In [2]:
#!pip install darts
!pip install optuna
!pip install optuna-dashboard

In [3]:
import optuna


/Users/sudiptahalder/Documents/school/TLS-Fingerprinting-Malicious-Detection/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import numpy as np
import json
import pandas as pd
import matplotlib.pyplot as plt
import requests
import os, random

In [5]:


#preprocessing
from sklearn.preprocessing import normalize
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import TargetEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import LabelBinarizer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline


#imbalance
from sklearn.impute import SimpleImputer
from imblearn.datasets import make_imbalance
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks

#datasets
from sklearn.datasets import load_iris
from imblearn.datasets import make_imbalance
from sklearn.datasets import make_classification
from sklearn.base import BaseEstimator, ClassifierMixin, RegressorMixin

from sklearn.model_selection import train_test_split

#imbalanced learning
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks

#models
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.neural_network import MLPClassifier
from sklearn.svm import OneClassSVM

#matrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import precision_recall_curve, auc
from sklearn.metrics import average_precision_score
from sklearn.metrics import adjusted_rand_score,normalized_mutual_info_score
from sklearn.metrics import silhouette_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix # imported:  it outputes fp,fn,tp,tn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc, ConfusionMatrixDisplay


from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.model_selection import cross_val_score


import seaborn as sns

In [6]:
import tensorflow as tf
from tensorflow import keras

from tensorflow.keras import layers, Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization,Dense, Conv2D, Flatten, Reshape
from tensorflow.keras.layers import Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import BatchNormalization


In [7]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))


Num GPUs Available:  1


In [8]:
seed = 2
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(
    seed
)
# Recommended for modern NumPy
# rng = np.random.default_rng(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

In [9]:
df = pd.read_csv('https://raw.githubusercontent.com/SoupBoi1/TLS-Fingerprinting-Malicious-Detection/refs/heads/main/dataset/dataset.csv')#getting the data

In [10]:

display(df.head())

,SrcPort,SNI,AppName,DstPort,JA3Shash,JA3hash,JA4hash,Type,SrcIP,Version,Filename,OrgName,DstIP,JA4Shash,Label,Attack
0,58688,officehymy.com,sodinokibi,443,61be9ce3d068c08ff99a857f62352f9d,a0e9f5d64349fb13191bc781f81f42e1,t12d190800_d83cc789557e_7af1ed941c26,M,10.127.0.109,0.0,tests/malware2/SODINOKIBI/240602-km899agd6t.be...,Japan Network Information Center; XSERVER20 (JP),157.112.183.48,t120400_c02f_460f64128655,SODINOKIBI,1
1,51187,sf16-muse-va.ibytedtos.com,star_quiz_01,443,42ec7b1db61428bf1cc6e01b9ef02b04,6ec2896feff5746955f700c0023f5804,t12d1409h1_c866b44c5a26_b39be8c56a14,M,192.168.0.100,0.0,tests/malware4/android11/star_quiz_01-extracte...,Slovak Telecom Network Administrator; BA-WEBHO...,212.5.219.10,t1206h1_c02c_e1dda4771ae8,MALWARE,1
2,61625,avatars.mds.yandex.net,zloader,443,4ee87de303b9c8138441e6527cbeea3e,cd08e31494f9531f560d64c695473da9,t13d1516h2_8daaf6152771_e5627efa2ab1,M,10.127.0.108,0.0,tests/malware2/ZLOADER/240525-adwbxsga76.behav...,YANDEX LLC; YANDEX-87-250-247 (RU),87.250.247.182,t1304h2_1302_a56c5b993250,ZLOADER,1
3,50972,js.stripe.com,asyncrat,443,60c22edca21e0598ad28f8d78c8dedcd,579ccef312d18482fc42e2b822ca2430,t13d1715h2_5b57614c22b0_a815a0c236aa,M,10.127.0.20,0.0,tests/malware2/ASYNCRAT/240608-e15lbsgh9x.beha...,Amazon.com; Inc.; AMAZON-CF,18.245.86.52,t1304h2_1301_a56c5b993250,ASYNCRAT,1
4,50579,rps-svcs.oracle.com,bazarbackdoor,443,a860078aa51e1102b117d1f8a1437b72,2a458dd9c65afbcf591cd8c2a194b804,t12d210600_b973bfd88a0e_1da50ec048a3,M,10.127.0.3,0.0,tests/malware2/BAZARBACKDOOR/230429-ytv8vsbh95...,Akamai International; BV; AIBV,23.222.50.60,t120400_c014_cbb8871a0652,BAZARBACKDOOR,1


In [11]:
df.info()#info
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30664 entries, 0 to 30663
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   SrcPort   30664 non-null  int64  
 1   SNI       30239 non-null  object 
 2   AppName   30664 non-null  object 
 3   DstPort   30664 non-null  int64  
 4   JA3Shash  28951 non-null  object 
 5   JA3hash   30664 non-null  object 
 6   JA4hash   30664 non-null  object 
 7   Type      30414 non-null  object 
 8   SrcIP     30664 non-null  object 
 9   Version   28951 non-null  float64
 10  Filename  28951 non-null  object 
 11  OrgName   30664 non-null  object 
 12  DstIP     30664 non-null  object 
 13  JA4Shash  28951 non-null  object 
 14  Label     29149 non-null  object 
 15  Attack    30664 non-null  int64  
dtypes: float64(1), int64(3), object(12)
memory usage: 3.7+ MB


,SrcPort,DstPort,Version,Attack
count,30664.000000,30664.000000,28951.0,30664.000000
mean,55442.169482,583.606933,0.0,0.787666
std,4529.980500,1485.352244,0.0,0.408966
min,33139.000000,80.000000,0.0,0.000000
25%,51291.750000,443.000000,0.0,1.000000
50%,54972.500000,443.000000,0.0,1.000000
75%,58615.250000,443.000000,0.0,1.000000
max,65533.000000,49816.000000,0.0,1.000000


In [12]:
df["DstIP"].unique()

array(['157.112.183.48', '212.5.219.10', '87.250.247.182', ...,
       '52.26.253.153', '2.22.89.33', '66.70.219.208'], dtype=object)

#Pre-possessing and Data Analysis

encoding and spliting the IPs

In [13]:
SrcIP_1=[]
SrcIP_2=[]
SrcIP_3=[]
SrcIP_4=[]
for i in df["SrcIP"]:

  #print(i.split('.'))
  SrcIP_1.append(int(i.split('.')[0]))
  SrcIP_2.append(int(i.split('.')[1]))
  SrcIP_3.append(int(i.split('.')[2]))
  SrcIP_4.append(int(i.split('.')[3]))
print(np.shape(SrcIP_1))
print(np.shape(SrcIP_2))
print(np.shape(SrcIP_3))
print(np.shape(SrcIP_4))

(30664,)
(30664,)
(30664,)
(30664,)


In [14]:
DstIP_1=[]
DstIP_2=[]
DstIP_3=[]
DstIP_4=[]
for i in df["DstIP"]:

  #print(i.split('.'))
  DstIP_1.append(int(i.split('.')[0]))
  DstIP_2.append(int(i.split('.')[1]))
  DstIP_3.append(int(i.split('.')[2]))
  DstIP_4.append(int(i.split('.')[3]))

jA4

In [15]:
def JA4_parameters_extration(JA4_A):
  protocal,TLS_version,SNI,Nciper,Nextension,ALPN=None,None,None,None,None,None
  if(len(JA4_A)==10):
    protocal=JA4_A[0]
    TLS_version=(JA4_A[1:3])
    SNI=JA4_A[3]
    Nciper=int(JA4_A[4:6])
    Nextension=int(JA4_A[6:7])
    ALPN=JA4_A[8:]
  return [protocal,TLS_version,SNI,Nciper,Nextension,ALPN]

def JA4S_parameters_extration(JA4S_A):
  protocal,TLS_version,Nextension,ALPN=None,None,None,None
  if(len(JA4S_A)==7):
    protocal=JA4S_A[0]
    TLS_version=(JA4S_A[1:3])
    Nextension=int(JA4S_A[3:5])
    ALPN=JA4S_A[5:]
  return [protocal,TLS_version,Nextension,ALPN]

def JA4H_parameters_extration(JA4H_A):
  method,version,cookie,ref,NHeaders,AL=None,None,None,None,None,None
  if(len(JA4H_A)==12):
    method=JA4H_A[0:2]
    version=(JA4H_A[2:4])
    cookie=JA4H_A[4]
    ref=JA4H_A[5]
    NHeaders=JA4H_A[6:8]
    AL=JA4H_A[8:]
  return [method,version,cookie,ref,NHeaders,AL]

print(np.array(JA4_parameters_extration('t13d1516h2')))
print(JA4S_parameters_extration('t120400'))
print(JA4H_parameters_extration('ge20cr13enus'))

['t' '13' 'd' '15' '1' 'h2']
['t', '12', 4, '00']
['ge', '20', 'c', 'r', '13', 'enus']


In [16]:
JA4A=[]
JA4B=[]
JA4C=[]
Protocal =[]
TLS_Version =[]
Nextension=[]
SNI= []
Nciper=[]
ALPN =[]

for i in df["JA4hash"]:
  array = JA4_parameters_extration(i.split('_')[0])
  Protocal.append(array[0])
  TLS_Version.append(array[1])
  SNI.append(array[2])
  Nciper.append(array[3])
  Nextension.append(array[4])
  ALPN.append(array[5])
  #print(i.split('.'))
  JA4A.append(i.split('_')[0])
  JA4B.append(i.split('_')[1])
  JA4C.append(i.split('_')[2])


In [17]:
JA4SA=[]
JA4SB=[]
JA4SC=[]
ProtocalS =[]
TLS_VersionS =[]
NextensionS=[]
ALPNS =[]

for i in df["JA4Shash"]:

  if i is not None and type(i)==str:
      array = JA4S_parameters_extration(i.split('_')[0])
      ProtocalS.append(array[0])
      TLS_VersionS.append(array[1])
      NextensionS.append(array[2])
      ALPNS.append(array[3])

      JA4SA.append(i.split('_')[0])
      JA4SB.append(i.split('_')[1])
      JA4SC.append(i.split('_')[2])
  else:
    #print(i)
    ProtocalS.append(None)
    TLS_VersionS.append(None)
    NextensionS.append(None)
    ALPNS.append(None)
    JA4SA.append(None)
    JA4SB.append(None)
    JA4SC.append(None)


In [18]:
le = LabelEncoder()

# 2. Fit and transform the column
df['Label'] = le.fit_transform(df['Label'])

# 3. View the mapping (optional)
y_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(f"Mapped classes: {y_mapping}")

Mapped classes: {'AGENTTESLA': 0, 'ASYNCRAT': 1, 'AZORULT': 2, 'BAZARBACKDOOR': 3, 'DARKCOMET': 4, 'DRIDEX': 5, 'EMOTET': 6, 'FORMBOOK': 7, 'GOZI_IFSB': 8, 'HAWKEYE': 9, 'HAWKEYE_REBORN': 10, 'ICEDID': 11, 'LOKIBOT': 12, 'MALWARE': 13, 'MASSLOGGER': 14, 'MATIEX': 15, 'METASPLOIT': 16, 'MODILOADER': 17, 'NANOCORE': 18, 'NETWIRE': 19, 'NJRAT': 20, 'NORMAL': 21, 'PONY': 22, 'QAKBOT': 23, 'QNODESERVICE': 24, 'RACCOON': 25, 'REMCOS': 26, 'REVENGERAT': 27, 'SMOKELOADER': 28, 'SODINOKIBI': 29, 'TRICKBOT': 30, 'UPATRE': 31, 'WANNACRY': 32, 'YUNSIP': 33, 'ZLOADER': 34, 'advertising/analytics': 35, nan: 36}


## 60/20/20 split


In [19]:


features_df = pd.DataFrame({

    'SrcIP_1': SrcIP_1, 'SrcIP_2': SrcIP_2, 'SrcIP_3': SrcIP_3, 'SrcIP_4': SrcIP_4,

    'DstIP_1': DstIP_1, 'DstIP_2': DstIP_2, 'DstIP_3': DstIP_3, 'DstIP_4': DstIP_4,

    'JA4_protocal':Protocal, 'JA4_TLS_version':TLS_Version,'JA4_SNI':SNI,'JA4_Nciper':Nciper ,'JA4_Nextension':Nextension , 'JA4_ALPN':ALPN, 'JA4B': JA4B, 'JA4C': JA4C,

     'JA4S_Protocal': ProtocalS, 'JA4S_TLS_Version': TLS_VersionS, 'JA4S_Nextension': NextensionS, 'JA4S_ALPNS': ALPNS, 'JA4SB': JA4SB, 'JA4SC': JA4SC,

    'SrcPort': df['SrcPort'],

    'DstPort': df['DstPort']#,'Attack': df['Attack']

})

In [20]:
hotencode_labels = ['JA4_TLS_version', 'JA4S_TLS_Version', 'JA4B', 'JA4C', 'JA4SB', 'JA4SC']

In [21]:

X_tempR, X_testR, y_tempR, y_test = train_test_split(features_df, df['Label'].values, test_size=0.20, random_state=42, stratify=df['Label'].values)

X_trainR, X_valR, y_train, y_val = train_test_split(X_tempR, y_tempR, test_size=0.25, random_state=42, stratify=y_tempR)

print(f"Training set size: {X_trainR.shape[0]}")

print(f"Validation set size: {X_valR.shape[0]}")

print(f"Testing set size: {X_testR.shape[0]}")

Training set size: 18398
Validation set size: 6133
Testing set size: 6133


In [22]:
X_trainR.columns

Index(['SrcIP_1', 'SrcIP_2', 'SrcIP_3', 'SrcIP_4', 'DstIP_1', 'DstIP_2',
       'DstIP_3', 'DstIP_4', 'JA4_protocal', 'JA4_TLS_version', 'JA4_SNI',
       'JA4_Nciper', 'JA4_Nextension', 'JA4_ALPN', 'JA4B', 'JA4C',
       'JA4S_Protocal', 'JA4S_TLS_Version', 'JA4S_Nextension', 'JA4S_ALPNS',
       'JA4SB', 'JA4SC', 'SrcPort', 'DstPort'],
      dtype='object')

In [23]:
X_trainR.select_dtypes(include=[np.number]).head()


,SrcIP_1,SrcIP_2,SrcIP_3,SrcIP_4,DstIP_1,DstIP_2,DstIP_3,DstIP_4,JA4_Nciper,JA4_Nextension,JA4S_Nextension,SrcPort,DstPort
7991,192,168,0,100,13,32,6,223,14.0,0.0,7.0,54571,443
23269,10,127,1,110,20,26,156,210,15.0,1.0,4.0,61307,443
14100,10,127,1,8,104,18,7,192,15.0,1.0,2.0,62250,443
19261,192,168,0,100,3,165,206,53,14.0,0.0,7.0,54540,443
27585,10,127,1,121,204,79,197,237,19.0,0.0,6.0,55224,443


In [24]:
X_trainR.loc[:, ~X_trainR.columns.isin(hotencode_labels)].select_dtypes(exclude=[np.number]).head()

,JA4_protocal,JA4_SNI,JA4_ALPN,JA4S_Protocal,JA4S_ALPNS
7991,t,d,h1,t,h1
23269,t,d,h2,t,h2
14100,t,d,h2,t,00
19261,t,d,h1,t,h1
27585,t,d,h2,t,h2


In [25]:
X_trainR.loc[:, X_trainR.columns.isin(hotencode_labels)].head()


,JA4_TLS_version,JA4B,JA4C,JA4S_TLS_Version,JA4SB,JA4SC
7991,12,c866b44c5a26,b39be8c56a14,12,c02f,2411d94cc0c2
23269,13,8daaf6152771,6cdcb247c39b,13,1301,a56c5b993250
14100,13,8daaf6152771,e5627efa2ab1,13,1301,234ea6891581
19261,12,c866b44c5a26,b39be8c56a14,12,c02f,2411d94cc0c2
27585,12,d83cc789557e,7af1ed941c26,12,c030,7136cef64a82


In [26]:
label_trainR = X_trainR.loc[:, ~X_trainR.columns.isin(hotencode_labels)].select_dtypes(exclude=[np.number])
label_testR = X_testR.loc[:, ~X_testR.columns.isin(hotencode_labels)].select_dtypes(exclude=[np.number])
label_valR = X_valR.loc[:, ~X_valR.columns.isin(hotencode_labels)].select_dtypes(exclude=[np.number])

In [27]:
hotencoding_trainR= X_trainR.loc[:, X_trainR.columns.isin(hotencode_labels)]
hotencoding_testR = X_testR.loc[:, X_testR.columns.isin(hotencode_labels)]
hotencoding_valR = X_valR.loc[:, X_valR.columns.isin(hotencode_labels)]

### encoding


#### numeric

In [28]:

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])


numeric_pipeline.fit(np.zeros([10,10]))


,steps,"[('imputer', ...), ('scaler', ...)]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,copy,True


In [29]:
X_train_num_t = X_trainR.select_dtypes(include=[np.number])
numeric_pipeline.fit(X_train_num_t)
X_train_num = numeric_pipeline.transform(X_train_num_t)

X_val_num_t = X_valR.select_dtypes(include=[np.number])
#numeric_pipeline.fit(X_val_num_t)
X_val_num = numeric_pipeline.transform(X_val_num_t)

X_test_num_t = X_testR.select_dtypes(include=[np.number])
#numeric_pipeline.fit(X_test_num_t)
X_test_num = numeric_pipeline.transform(X_test_num_t)


#### *categorical*

In [30]:
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

In [31]:
categorical_pipeline.fit(hotencoding_trainR)
X_train_cat = categorical_pipeline.transform(hotencoding_trainR)


X_val_cat = categorical_pipeline.transform(hotencoding_testR)

X_test_cat = categorical_pipeline.transform(hotencoding_valR)

#### Label based

In [32]:
label_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder())
])


In [33]:
label_pipeline.fit(label_trainR)
X_train_label = label_pipeline.transform(label_trainR)

X_val_label = label_pipeline.transform(label_testR)

X_test_label = label_pipeline.transform(label_valR)


In [34]:
X_train = np.concatenate((X_train_num, X_train_cat.toarray(),X_train_label), axis=1)
X_val = np.concatenate((X_val_num, X_val_cat.toarray(),X_val_label), axis=1)
X_test = np.concatenate((X_test_num, X_test_cat.toarray(),X_test_label), axis=1)


In [ ]:
# ── SMOTE: Balance the training set (applied ONLY to training data) ──
print("Class distribution before SMOTE:", dict(zip(*np.unique(y_train, return_counts=True))))

smote = SMOTE(random_state=42, k_neighbors=1)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Class distribution after SMOTE: ", dict(zip(*np.unique(y_train_resampled, return_counts=True))))
print(f"X_train shape before SMOTE: {X_train.shape} → after: {X_train_resampled.shape}")


Class distribution before SMOTE: {0: 222, 1: 310, 2: 142, 3: 145, 4: 20, 5: 89, 6: 32, 7: 45, 8: 2, 9: 391, 10: 95, 11: 242, 12: 109, 13: 3095, 14: 664, 15: 115, 16: 133, 17: 151, 18: 166, 19: 162, 20: 74, 21: 3907, 22: 11, 23: 104, 24: 49, 25: 172, 26: 32, 27: 403, 28: 214, 29: 539, 30: 101, 31: 65, 32: 970, 33: 62, 34: 1958, 35: 2498, 36: 909}


ValueError: Expected n_neighbors <= n_samples_fit, but n_neighbors = 6, n_samples_fit = 2, n_samples = 2

# Model Cration, Training, Validation,Testing

In [36]:
storage_name = "sqlite:///optuna_study.db"


In [37]:
D= np.shape(X_train)[1]

In [38]:
B =[16, 32, 64]
ab = [(1,1),(0.5,1),(0.1,1)]

In [39]:
# gemini generated code
def modelSuperAutoencoder(input_size=16, bottleneck_dim=32):
  # --- encode ---
  inputs = keras.Input(shape=(input_size,))
  d1 = layers.Dense(128, activation='relu')(inputs)
  d2 = layers.Dense(64, activation='relu')(d1)

  bottleneck = layers.Dense(bottleneck_dim, activation='relu', name="bottleneck")(d2)

  # --- Decoder ---

  d3 = layers.Dense(64, activation='relu',)(bottleneck)
  d4 = layers.Dense(128, activation='relu')(d3)
  x_recon = layers.Dense(input_size, activation='linear', name="x_recon")(d4)

  # --- Classifier Head ---

  d5 = layers.Dense(32, activation='relu')(bottleneck)
  y_pred = layers.Dense(1, activation='sigmoid', name="y_pred")(d5)

  # --- Full Model ---
  model = Model(inputs=inputs, outputs=[x_recon, y_pred])

  return model

In [ ]:
def modelSuperAutoencoder_objective(trial):
    B = trial.suggest_int("B", 16, 32, 64)
    ab_str = trial.suggest_categorical("alpha beta", ["(1,1)","(0.5,1)","(0.1,1)"])
    ab = eval(ab_str)
    model_name = f"saved_models/modelSuperAutoencoder/modelSuperAutoencoder_B_{B}_alpha_beta_{ab_str}.keras"

    if os.path.exists(model_name):
        print(f"Loading existing model: {model_name}")
        model = keras.models.load_model(model_name)
    else:

        model = modelSuperAutoencoder(input_size = D,bottleneck_dim=B)
        model.compile(
        optimizer='adam',
        loss={
            'x_recon': 'mse',
            'y_pred': 'binary_crossentropy'
        },
        loss_weights={
            'x_recon': ab[0],
            'y_pred': ab[1]
        },
        metrics={'y_pred': 'accuracy'}
        )
        callback = keras.callbacks.EarlyStopping(monitor='val_y_pred_loss',patience=5,restore_best_weights=True, mode='min')

        model.fit(X_train, {"x_recon": X_train, "y_pred": y_train}, callbacks=callback , epochs=50 ,batch_size = 256,verbose=0)
    
    os.makedirs('saved_models/modelSuperAutoencoder', exist_ok=True)
    model.save(model_name)
    print(f"      Saved model to: {model_name}")



    p_val= model.predict(X_val)
    pred = (p_val[1] > 0.5).astype(int)
    p_val = np.reshape(p_val[1],(-1))
    pred =np.reshape(pred,(-1))

    #acc = accuracy_score(y_val, pred)
    #prec = precision_score(y_val, pred, zero_division=0)
    #rec = recall_score(y_val, pred, zero_division=0)
    f1 = f1_score(y_val, pred, average='weighted', zero_division=0)

MSAE_study = optuna.create_study(
    study_name="modelSuperAutoencoder",
    storage=storage_name,
    direction="maximize",
    load_if_exists=True
)

MSAE_study.optimize(modelSuperAutoencoder_objective, n_trials=9)

#print(f"Best Accuracy: {MSAE_study.best_value}")
#print(f"Best Params: {MSAE_study.best_params}")

[I 2026-05-03 00:41:40,352] Using an existing study with name 'modelSuperAutoencoder' instead of creating a new one.
/var/folders/n9/hjzg_lkj5n1bp1v60850ymbh0000gn/T/ipykernel_40989/1362822154.py:2: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  B = trial.suggest_int("B", 16, 32, 64)


Loading existing model: saved_models/modelSuperAutoencoder/modelSuperAutoencoder_B_16_alpha_beta_(0.5,1).keras
      Saved model to: saved_models/modelSuperAutoencoder/modelSuperAutoencoder_B_16_alpha_beta_(0.5,1).keras
192/192 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


[W 2026-05-03 00:41:41,040] Trial 4 failed with parameters: {'B': 16, 'alpha beta': '(0.5,1)'} because of the following error: The value None could not be cast to float..
[W 2026-05-03 00:41:41,041] Trial 4 failed with value None.
/var/folders/n9/hjzg_lkj5n1bp1v60850ymbh0000gn/T/ipykernel_40989/1362822154.py:2: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  B = trial.suggest_int("B", 16, 32, 64)
/Users/sudiptahalder/Documents/school/TLS-Fingerprinting-Malicious-Detection/.venv/lib/python3.10/site-packages/optuna/distributions.py:684: UserWarning: The distribution is specified by [16, 32] and step=64, b

      Saved model to: saved_models/modelSuperAutoencoder/modelSuperAutoencoder_B_16_alpha_beta_(1,1).keras
192/192 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


[W 2026-05-03 00:43:07,416] Trial 5 failed with parameters: {'B': 16, 'alpha beta': '(1,1)'} because of the following error: The value None could not be cast to float..
[W 2026-05-03 00:43:07,416] Trial 5 failed with value None.


...

Loading existing model: saved_models/modelSuperAutoencoder/modelSuperAutoencoder_B_16_alpha_beta_(1,1).keras
      Saved model to: saved_models/modelSuperAutoencoder/modelSuperAutoencoder_B_16_alpha_beta_(1,1).keras
  1/192 ━━━━━━━━━━━━━━━━━━━━ 6s 34ms/step

/var/folders/n9/hjzg_lkj5n1bp1v60850ymbh0000gn/T/ipykernel_40989/1362822154.py:2: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  B = trial.suggest_int("B", 16, 32, 64)
/Users/sudiptahalder/Documents/school/TLS-Fingerprinting-Malicious-Detection/.venv/lib/python3.10/site-packages/optuna/distributions.py:684: UserWarning: The distribution is specified by [16, 32] and step=64, but the range is not divisible by `step`. It will be replaced with [16, 16].
  optuna_warn(


192/192 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


[W 2026-05-03 00:44:38,724] Trial 12 failed with parameters: {'B': 16, 'alpha beta': '(1,1)'} because of the following error: The value None could not be cast to float..
[W 2026-05-03 00:44:38,724] Trial 12 failed with value None.


ValueError: Record does not exist.

In [ ]:
def modelSuperAutoencoder_objective_smote(trial):
    B = trial.suggest_int("B", 16, 32, 64)
    ab_str = trial.suggest_categorical("alpha beta", ["(1,1)","(0.5,1)","(0.1,1)"])
    ab = eval(ab_str)

    model = modelSuperAutoencoder(input_size = D,bottleneck_dim=B)
    model.compile(
    optimizer='adam',
    loss={
        'x_recon': 'mse',
        'y_pred': 'binary_crossentropy'
    },
    loss_weights={
        'x_recon': ab[0],
        'y_pred': ab[1]
    },
    metrics={'y_pred': 'accuracy'}
    )
    callback = keras.callbacks.EarlyStopping(monitor='val_y_pred_loss',patience=5,restore_best_weights=True, mode='min')

    model.fit(X_train_resampled, {"x_recon": X_train_resampled, "y_pred": y_train_resampled}, callbacks=callback , epochs=50 ,batch_size = 256,verbose=0)
    
    os.makedirs('savedSMOTE_models/modelSuperAutoencoder', exist_ok=True)
    model_name = f"savedSMOTE_models/modelSuperAutoencoder/modelSuperAutoencoder_B_{B}_alpha_beta_{ab_str}_t{trial.number}.keras"
    model.save(model_name)
    print(f"      Saved model to: {model_name}")



    p_val= model.predict(X_val)
    pred = (p_val[1] > 0.5).astype(int)
    p_val = np.reshape(p_val[1],(-1))
    pred =np.reshape(pred,(-1))

    #acc = accuracy_score(y_val, pred)
    #prec = precision_score(y_val, pred, zero_division=0)
    #rec = recall_score(y_val, pred, zero_division=0)
    f1 = f1_score(y_val, pred, zero_division=0)

MSAE_study = optuna.create_study(
    study_name="modelSuperAutoencoder",
    storage=storage_name,
    direction="maximize",
    load_if_exists=True
)

MSAE_study.optimize(modelSuperAutoencoder_objective, n_trials=9)

print(f"Best Accuracy: {MSAE_study.best_value}")
print(f"Best Params: {MSAE_study.best_params}")